# C-DIRA Walkthrough

This notebook runs the full C-DIRA pipeline one stage at a time and visualizes
each part: preprocessing, pseudo-domains, model architecture, live training,
and evaluation.

**Requirements:** a CUDA GPU runtime (Runtime > Change runtime type > GPU) and a
Kaggle `kaggle.json` API token for an account that has accepted the *State Farm
Distracted Driver Detection* competition rules. Run the cells top to bottom from
the repository root.


In [1]:
import subprocess
import sys
from pathlib import Path

assert Path("pyproject.toml").exists() and Path("src/cdira").is_dir(), (
    "Run this notebook from the C-DIRA repository root."
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)


/Users/hoangtrung1801/develpoment/0-AI/driver-behavior-detection/.venv/bin/python: No module named pip


CalledProcessError: Command '['/Users/hoangtrung1801/develpoment/0-AI/driver-behavior-detection/.venv/bin/python', '-m', 'pip', 'install', '-q', '-e', '.']' returned non-zero exit status 1.

## Stage 0 — Environment

Verify the repository, Python 3.12, and a usable CUDA GPU, then load the Colab
experiment config.


In [2]:
import time

import matplotlib.pyplot as plt

from cdira import colab
from cdira.config import load_config

ROOT = colab.ensure_repository_root()
colab.ensure_supported_python()
DEVICE = colab.ensure_cuda_available()
CONFIG = load_config(Path("configs/colab.yaml"))
print("Repository:", ROOT, "| Device:", DEVICE)


ColabSetupError: This notebook requires a CUDA GPU runtime. In Colab choose Runtime > Change runtime type > GPU, then rerun.

In [ ]:
colab.install_kaggle_credentials()


## Stage 1 — Dataset

Reuse a valid existing dataset, or download and extract the competition data.
Access errors are translated into an actionable message.


In [ ]:
from cdira.data.download import (
    DatasetLayoutError,
    download_competition,
    validate_dataset,
)

started = time.perf_counter()
try:
    fingerprint = validate_dataset(CONFIG.paths.data_root)
    print("Reusing dataset:", fingerprint.image_count, "images")
except DatasetLayoutError:
    try:
        fingerprint = download_competition(CONFIG.paths.data_root)
    except Exception as error:  # noqa: BLE001
        raise SystemExit(colab.translate_kaggle_error(error)) from error
    print("Downloaded:", fingerprint.image_count, "images")
print(f"elapsed={time.perf_counter() - started:.1f}s")


## Stage 2 — Preprocessing & data

Build the stratified splits, then show class balance, split sizes, sample
images, and the augmentation applied during training.


In [ ]:
import pandas as pd

from cdira.data.manifests import build_split_manifests
from cdira.reporting import figures

bundle = build_split_manifests(
    CONFIG.paths.data_root, CONFIG.paths.manifest_root, CONFIG.seed
)
frames = {
    "train": pd.read_csv(bundle.train_path),
    "validation": pd.read_csv(bundle.validation_path),
    "test": pd.read_csv(bundle.test_path),
}
sizes = {name: len(frame) for name, frame in frames.items()}
counts = frames["train"]["class_id"].value_counts().sort_index().to_dict()
display(figures.split_sizes_table(sizes))
figures.class_distribution_figure(counts)
plt.show()


In [ ]:
from PIL import Image

from cdira.data.dataset import build_transform

samples, labels = [], []
for class_id, group in frames["train"].groupby("class_id"):
    row = group.iloc[0]
    samples.append(Image.open(CONFIG.paths.data_root / row.relative_path).convert("RGB"))
    labels.append(f"c{class_id}")
figures.sample_grid_figure(samples, labels, ncols=5, title="One sample per class")
plt.show()

row = frames["train"].iloc[0]
original = Image.open(CONFIG.paths.data_root / row.relative_path).convert("RGB")
train_tensor = build_transform(
    True, CONFIG.data.image_size, CONFIG.data.horizontal_flip, CONFIG.data.brightness
)(original)
eval_tensor = build_transform(False, CONFIG.data.image_size, False, CONFIG.data.brightness)(
    original
)
figures.augmentation_figure(original, train_tensor, eval_tensor)
plt.show()


## Stage 3 — Pseudo-domains

Cluster backbone features into pseudo-domains. Show the silhouette-based choice
of k, a 2-D PCA projection colored by domain, domain sizes, and a sample per
domain.


In [ ]:
from cdira.pipeline import prepare_domains

prepared = prepare_domains(CONFIG)
domains = prepared.domains
print("Selected k =", domains.k)

figures.silhouette_figure(domains.silhouette_scores, domains.k)
plt.show()

train_cache = prepared.caches["train"]
figures.domain_scatter_figure(
    train_cache.features,
    [domains.labels_by_path[path] for path in train_cache.paths],
)
plt.show()

display(figures.domain_sizes_table(domains.labels_by_path))

first_by_domain: dict[int, str] = {}
for path, domain_id in domains.labels_by_path.items():
    first_by_domain.setdefault(domain_id, path)
domain_images = [
    Image.open(CONFIG.paths.data_root / path).convert("RGB")
    for path in first_by_domain.values()
]
figures.sample_grid_figure(
    domain_images,
    [f"domain {domain_id}" for domain_id in first_by_domain],
    ncols=5,
    title="Representative image per domain",
)
plt.show()


## Stage 4 — Model architecture

Instantiate C-DIRA and inspect its branch structure and parameter budget.


In [ ]:
from cdira.models.cdira import CDIRA

model = CDIRA(
    num_classes=CONFIG.model.num_classes,
    num_domains=domains.k,
    top_k=CONFIG.model.top_k,
    global_hidden=CONFIG.model.global_hidden,
    roi_hidden=CONFIG.model.roi_hidden,
    fused_hidden=CONFIG.model.fused_hidden,
    routing_hidden=CONFIG.model.routing_hidden,
    domain_hidden=CONFIG.model.domain_hidden,
    grl_strength=CONFIG.model.grl_strength,
    pretrained=True,
)
figures.architecture_schematic_figure()
plt.show()
display(figures.parameter_table(model))


## Stage 5 — Training

Train C-DIRA with a live loss curve, then train the MobileNetV3 baseline.
Checkpoints are written to a fresh timestamped run under `artifacts/colab`.


In [ ]:
import torch
from IPython.display import clear_output
from torch.utils.data import DataLoader

from cdira.artifacts import RunArtifacts
from cdira.data.dataset import StateFarmDataset
from cdira.models.cdira import MobileNetBaseline
from cdira.pipeline import domain_mapping
from cdira.runtime import select_device
from cdira.training.engine import Trainer
from cdira.training.losses import LossWeights

domain_ids = domain_mapping(domains)


def make_loader(manifest: Path, training: bool) -> DataLoader:
    transform = build_transform(
        training,
        CONFIG.data.image_size,
        CONFIG.data.horizontal_flip if training else False,
        CONFIG.data.brightness,
    )
    dataset = StateFarmDataset(manifest, CONFIG.paths.data_root, transform, domain_ids)
    return DataLoader(
        dataset,
        batch_size=CONFIG.training.batch_size,
        shuffle=training,
        num_workers=CONFIG.data.num_workers,
    )


train_loader = make_loader(bundle.train_path, True)
validation_loader = make_loader(bundle.validation_path, False)
test_loader = make_loader(bundle.test_path, False)

run = RunArtifacts.create(CONFIG, run_id=time.strftime("colab-%Y%m%d-%H%M%S"))
device = select_device(CONFIG.training.device)
weights = LossWeights(**CONFIG.training.loss_weights.model_dump())
trainer = Trainer(
    model,
    device,
    CONFIG.training.max_epochs,
    CONFIG.training.patience,
    CONFIG.training.learning_rate,
    weights,
    CONFIG.training.confidence_threshold,
)

curve_figure, curve_axis = plt.subplots(figsize=(6, 4))


def update_curve(history: list[dict[str, float]]) -> None:
    figures.draw_loss_curve(curve_axis, history)
    clear_output(wait=True)
    display(curve_figure)


trainer.fit(train_loader, validation_loader, on_epoch_end=update_curve)
torch.save(model.state_dict(), run.root / "checkpoints" / "cdira.pt")

baseline = MobileNetBaseline(CONFIG.model.num_classes, pretrained=True)
Trainer(
    baseline,
    device,
    CONFIG.training.max_epochs,
    CONFIG.training.patience,
    CONFIG.training.learning_rate,
).fit(train_loader, validation_loader)


## Stage 6 — Evaluation & interpretability

Confusion matrix, per-class F1, routing usage, the C-DIRA-vs-baseline
comparison, and ROI saliency overlays on a few test images.


In [ ]:
import json

from cdira.evaluation.metrics import classification_metrics
from cdira.evaluation.predict import collect_predictions
from cdira.models.cdira import RoutingPolicy
from cdira.pipeline import _baseline_predictions

table = collect_predictions(
    model, test_loader, RoutingPolicy.HEAD, CONFIG.routing.threshold, device
)
metrics = classification_metrics(table, CONFIG.model.num_classes)
baseline_table = _baseline_predictions(baseline, test_loader, device)
baseline_metrics = classification_metrics(baseline_table, CONFIG.model.num_classes)
(run.root / "metrics" / "full.json").write_text(json.dumps(metrics, indent=2))
(run.root / "metrics" / "baseline.json").write_text(json.dumps(baseline_metrics, indent=2))

figures.confusion_matrix_figure(metrics["confusion_matrix"])
plt.show()
figures.per_class_f1_figure(metrics["per_class"])
plt.show()
figures.routing_usage_figure(metrics)
plt.show()
figures.model_comparison_figure(metrics, baseline_metrics)
plt.show()


In [ ]:
from cdira.evaluation.visualize import roi_overlay_figure

batch = next(iter(test_loader))
images = batch["image"][:4].to(device)
output = model.predict(images, RoutingPolicy.HEAD, CONFIG.routing.threshold)
for index in range(images.shape[0]):
    raw = Image.open(CONFIG.paths.data_root / batch["relative_path"][index]).convert("RGB")
    title = f"true={int(batch['target'][index])} pred={int(output.logits[index].argmax())}"
    roi_overlay_figure(raw, output.saliency[index], output.topk_indices[index], title)
    plt.show()


## Stage 7 — Reports & artifacts

Generate the Markdown and self-contained HTML reports and print the artifact
paths. In Colab the HTML report is offered as a download.


In [ ]:
from cdira.reporting.html_report import build_html_reproduction_report
from cdira.reporting.report import build_reproduction_report

markdown_path = build_reproduction_report(run.root)
html_path = build_html_reproduction_report(run.root)
print("Run directory:", run.root)
print("Checkpoint:", run.root / "checkpoints" / "cdira.pt")
print("Markdown report:", markdown_path)
print("HTML report:", html_path)
try:
    from google.colab import files  # type: ignore[import-not-found]

    files.download(str(html_path))
except Exception:  # noqa: BLE001
    pass
